# Fine-tune mắt và ngáp nhiều góc trên Kaggle GPU

Gói chứa crop camera từ hai clip train; clip **WIN_20261013_17_07_01_Pro** chỉ dùng cho test sau khi chọn checkpoint. Validation là dữ liệu ảnh cũ, không dùng frame test để chọn epoch/ngưỡng. Cùng một người/buổi quay, chưa phải kiểm thử độc lập theo người.

Model bắt đầu từ eye_refined_v4 và yawn_kaggle_near_medium. Fine-tune Classify conv/BN affine/linear, backbone/BN running buffers cố định. Augmentation dịch crop, độ sáng, downsampling và blur nhẹ trước cùng preprocessing 224×224. Frame mắt khó đọc loại khỏi train. Hai độ phân giải từ cùng frame kế thừa một split và không được coi là hai mẫu độc lập.

GPU được yêu cầu; notebook dừng nếu không có CUDA. Huấn luyện chạy trong **Python process mới** sau khi cài dependency để tránh kernel giữ NumPy phiên bản cũ trong bộ nhớ. Output gồm model PT/ONNX, manifest, thống kê trước/sau, confusion matrix và UNKNOWN theo confidence/chất lượng. Các ngưỡng runtime giữ nguyên; chưa có nghiệm thu camera trực tiếp/NPU.


In [ ]:
import sys, subprocess, os
from pathlib import Path
from importlib.metadata import version
assert (3,9) <= sys.version_info[:2] < (3,14), 'Verify binary wheels for this Python version.'
os.environ['YOLO_CONFIG_DIR']='/kaggle/working/.config/ultralytics'
os.environ['MPLCONFIGDIR']='/kaggle/working/.config/matplotlib'
requirements=Path('/kaggle/working/requirements-multiview.txt')
requirements.write_text('# Keep Kaggle\'s CUDA-enabled torch/torchvision; do not replace them with CPU wheels.\nultralytics==8.4.163\nonnx==1.17.0; python_version < "3.13"\nonnx==1.19.1; python_version >= "3.13"\nonnxruntime==1.20.1; python_version < "3.13"\nonnxruntime==1.23.2; python_version >= "3.13"\nnumpy==1.26.4; python_version < "3.13"\nnumpy==2.2.6; python_version >= "3.13"\nPillow==11.1.0\nopencv-python==4.11.0.86\n',encoding='utf-8')
constraints=Path('/kaggle/working/gpu-constraints.txt')
constraints.write_text(''.join(f'{name}=={version(name)}\n' for name in ('torch','torchvision')),encoding='utf-8')
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--only-binary=:all:','-c',str(constraints),'-r',str(requirements)])


In [ ]:
import json
matches=[path for path in Path('/kaggle/input').rglob('bundle.json') if json.loads(path.read_text()).get('kind')=='driver_monitor_multiview']
assert len(matches)==1, 'Attach exactly one multiview dataset.'
bundle_root=matches[0].parent
print('Dataset:',bundle_root)
subprocess.check_call([sys.executable,'-u',str(bundle_root/'kaggle_multiview_training.py'),
                      '--root',str(bundle_root),'--output','/kaggle/working/multiview_model',
                      '--epochs','30','--batch-size','64','--patience','7'])


In [ ]:
summary=json.loads(Path('/kaggle/working/multiview_model/summary.json').read_text())
print(json.dumps(summary,indent=2))
from IPython.display import display,Image
for path in sorted(Path('/kaggle/working/multiview_model').glob('*letterbox_640_matrix.png')):
    print(path.name)
    display(Image(filename=str(path)))
print('Download /kaggle/working/multiview_result.zip')
